# 00 — Exploração das fontes

## Objetivo

Este notebook realiza a exploração inicial das fontes disponibilizadas para o case técnico de Engenharia de Dados.

Nesta etapa serão executadas as seguintes atividades:

- identificação e validação dos arquivos recebidos;
- disponibilização das fontes em uma área de landing;
- leitura dos diferentes formatos;
- inspeção de schemas e estruturas;
- contagem de registros e colunas;
- identificação inicial de duplicidades;
- avaliação de campos nulos;
- investigação dos possíveis relacionamentos entre as fontes;
- registro dos problemas de qualidade encontrados.

## Ações

- Nenhum dado corrigido nesta etapa.

- Os dados serão preservados como recebidos.
- Conversões e padronizações serão realizados na camada Silver.
- Registros inválidos deverão vão ser sinalizados, e não apenas descartados.
- Toda decisão de tratamento vai ser documentada e justificado.


In [0]:
# Precisa instalar essa dependencia não é nativa do Databricks
%pip install openpyxl

In [0]:
import os
import shutil
from pathlib import Path
from datetime import datetime

import pandas as pd

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

print(f"Diretório do notebook: {os.getcwd()}")
print(f"Versão do Spark: {spark.version}")

In [0]:
ARQUIVOS_ESPERADOS = [
    "atendimento_ocorrencias.ndjson",
    "cadastro_produtos_api_dump.json",
    "comercial_canais.xlsx",
    "crm_clientes_export.xlsx",
    "erp_pedidos_cabecalho_2025.csv",
    "erp_pedidos_itens_2025.csv",
    "legado_regioes_pipe.txt",
    "logistica_entregas.json",
    "vendedores.csv"
]

print(f"Quantidade de fontes esperadas: {len(ARQUIVOS_ESPERADOS)}")

In [0]:
from pathlib import Path
import os

# Captura do user conectado
USUARIO_ATUAL = spark.sql(
    "SELECT current_user() AS usuario"
).first()["usuario"]

DIRETORIO_ATUAL = Path(os.getcwd())

# Raízes do projeto
RAIZES_DE_BUSCA = [
    DIRETORIO_ATUAL,
    DIRETORIO_ATUAL.parent,
    Path("/Workspace/Users") / USUARIO_ATUAL,
    Path("/Workspace/Repos") / USUARIO_ATUAL
]

print(f"Usuário atual: {USUARIO_ATUAL}")
print(f"Diretório atual: {DIRETORIO_ATUAL}")

# Conta quantos arquivos esperados existem no diretório informado.
def contar_arquivos_esperados(diretorio: Path) -> int:
    return sum(
        (diretorio / arquivo).is_file()
        for arquivo in ARQUIVOS_ESPERADOS
    )

# Procura pastas chamadas datasources nas raízes
def localizar_pastas_datasources(
    raizes: list[Path]
) -> list[Path]:
    encontrados = []

    for raiz in raizes:
        if not raiz.exists():
            continue

        # Se a própria raiz for o datasources
        if raiz.name.lower() == "caseengenheiro":
            encontrados.append(raiz)

        # Caso o datasources ficar abaixo da raiz
        candidato_direto = raiz / "caseengenheiro"

        if candidato_direto.is_dir():
            encontrados.append(candidato_direto)

        # Pesquisa recursiva dentro da raiz
        try:
            for candidato in raiz.rglob("caseengenheiro"):
                if candidato.is_dir():
                    encontrados.append(candidato)
        # Vou ignorar o diretórios sem permissão de leitura
        except (PermissionError, OSError):
            continue

    # Removendo caminhos duplicados
    unicos = []

    for caminho in encontrados:
        caminho_resolvido = str(caminho)

        if caminho_resolvido not in [
            str(item) for item in unicos
        ]:
            unicos.append(caminho)
    return unicos


pastas_encontradas = localizar_pastas_datasources(
    RAIZES_DE_BUSCA
)

if not pastas_encontradas:
    raise FileNotFoundError(
        "Nenhuma pasta chamada 'datasources' foi encontrada nas áreas de trabalho"
    )


# Validando em qual pasta estão os arquivos da lista
resultado_busca = [
    (
        pasta,
        contar_arquivos_esperados(pasta)
    )
    for pasta in pastas_encontradas
]

resultado_busca = sorted(
    resultado_busca,
    key=lambda item: item[1],
    reverse=True
)

print("\nPastas caseengenheiro encontradas:")

for pasta, quantidade in resultado_busca:
    print(
        f"- {pasta} "
        f"({quantidade}/{len(ARQUIVOS_ESPERADOS)} arquivos)"
    )


# Seleciona a pasta
SOURCE_DIR = resultado_busca[0][0]

arquivos_ausentes = [
    arquivo
    for arquivo in ARQUIVOS_ESPERADOS
    if not (SOURCE_DIR / arquivo).is_file()
]

print(f"\nDiretório selecionado: {SOURCE_DIR}")
print(
    f"Arquivos encontrados: "
    f"{len(ARQUIVOS_ESPERADOS) - len(arquivos_ausentes)}"
    f"/{len(ARQUIVOS_ESPERADOS)}"
)

if arquivos_ausentes:
    raise FileNotFoundError(
        "A pasta datasources foi encontrada, mas está incompleta.\n\n"
        f"Diretório avaliado: {SOURCE_DIR}\n\n"
        "Arquivos ausentes:\n- "
        + "\n- ".join(arquivos_ausentes)
    )

print("\nTodas as fontes foram localizadas com sucesso!!!")

In [0]:
# ============================================================
# CONFIGURAÇÃO DA ÁREA DE LANDING
# ============================================================

SCHEMA_CASE = "case_data_engineer"
VOLUME_LANDING = "landing"

# Catalogo selecionado
catalogo_atual = spark.sql(
    "SELECT current_catalog() AS catalogo"
).first()["catalogo"]

# Catálogos disponíveis
catalogos_disponiveis = [
    linha["catalog"]
    for linha in spark.sql("SHOW CATALOGS").collect()
]

# Catálogos PROIBIDOS de receber objetos do projeto
catalogos_ignorados = {
    "system",
    "samples",
    "hive_metastore"
}

# Prioriza o catálogo atual e depois veirica os demais
catalogos_candidatos = []

if (
    catalogo_atual
    and catalogo_atual.lower() not in catalogos_ignorados
):
    catalogos_candidatos.append(catalogo_atual)

for catalogo in catalogos_disponiveis:
    if (
        catalogo.lower() not in catalogos_ignorados
        and catalogo not in catalogos_candidatos
    ):
        catalogos_candidatos.append(catalogo)

if not catalogos_candidatos:
    raise RuntimeError(
        "Nenhum catálogo do Unity Catalog foi encontrado."
    )

print(f"Catálogo atual: {catalogo_atual}")
print(f"Catálogos candidatos: {catalogos_candidatos}")

# ============================================================
# CONFIGURAÇÃO DO CATAALOGO
# ============================================================

CATALOG = None
erros_catalogos = {}

for catalogo in catalogos_candidatos:
    try:
        spark.sql(
            f"""
            CREATE SCHEMA IF NOT EXISTS
            `{catalogo}`.`{SCHEMA_CASE}`
            COMMENT 'Objetos do case técnico de Engenharia de Dados'
            """
        )

        CATALOG = catalogo
        break

    except Exception as erro:
        erros_catalogos[catalogo] = str(erro)[:500]

if CATALOG is None:
    detalhes = "\n".join(
        f"- {catalogo}: {mensagem}"
        for catalogo, mensagem in erros_catalogos.items()
    )

    raise RuntimeError(
        "Sem permissão de criar o schema no catálogo.\n"
        f"{detalhes}"
    )

print(f"Catálogo selecionado: {CATALOG}")
print(f"Schema selecionado: {SCHEMA_CASE}")


# ============================================================
# CONFIGURAÇÃO DO VOLUME
# ============================================================

spark.sql(
    f"""
    CREATE VOLUME IF NOT EXISTS
    `{CATALOG}`.`{SCHEMA_CASE}`.`{VOLUME_LANDING}`
    COMMENT 'Área de landing das fontes brutas do case'
    """
)

VOLUME_PATH = (
    f"/Volumes/"
    f"{CATALOG}/"
    f"{SCHEMA_CASE}/"
    f"{VOLUME_LANDING}"
)

print("\nEstrutura criada com sucesso:")
print(f"Catálogo: {CATALOG}")
print(f"Schema:   {SCHEMA_CASE}")
print(f"Volume:   {VOLUME_LANDING}")
print(f"Caminho:  {VOLUME_PATH}")

In [0]:
# ============================================================
# COPIAR OS ARQUIVOS JOGANDO PRA LANDING
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType
)

os.makedirs(VOLUME_PATH, exist_ok=True)

resultado_copia = []

for nome_arquivo in ARQUIVOS_ESPERADOS:

    origem = SOURCE_DIR / nome_arquivo
    destino = Path(VOLUME_PATH) / nome_arquivo

    tamanho_origem = origem.stat().st_size

    if (
        destino.exists()
        and destino.stat().st_size == tamanho_origem
    ):
        status = "JÁ EXISTIA"

    else:
        shutil.copyfile(
            src=str(origem),
            dst=str(destino)
        )

        status = "COPIADO"

    resultado_copia.append(
        (
            nome_arquivo,
            tamanho_origem,
            str(origem),
            str(destino),
            status
        )
    )

schema_copia = StructType([
    StructField("arquivo", StringType(), False),
    StructField("tamanho_bytes", LongType(), False),
    StructField("origem", StringType(), False),
    StructField("destino", StringType(), False),
    StructField("status", StringType(), False)
])

df_resultado_copia = spark.createDataFrame(
    resultado_copia,
    schema=schema_copia
)

display(
    df_resultado_copia.orderBy("arquivo")
)

In [0]:
# ============================================================
# DOUBLE CHECK NOS ARQUIVOS
# ============================================================

arquivos_no_volume = [
    arquivo.name
    for arquivo in Path(VOLUME_PATH).iterdir()
    if arquivo.is_file()
]

arquivos_volume_ausentes = sorted(
    set(ARQUIVOS_ESPERADOS)
    - set(arquivos_no_volume)
)

if arquivos_volume_ausentes:
    raise RuntimeError(
        "Deu ruim, arquivos que não foram:\n- "
        + "\n- ".join(arquivos_volume_ausentes)
    )

print(
    f"Landing validada: "
    f"{len(ARQUIVOS_ESPERADOS)}/"
    f"{len(ARQUIVOS_ESPERADOS)} arquivos disponíveis."
)

In [0]:
# ============================================================
# FUNÇÕES DE LEITURA DOS DADOS BRUTAS
# ============================================================

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType
)

# Validando o caminho completo do arquivo no landing.
def caminho_landing(nome_arquivo: str) -> str:
    return f"{VOLUME_PATH}/{nome_arquivo}"

# Adicionando informações de origem ao DataFrame.
def adicionar_metadados(
    df: DataFrame,
    nome_arquivo: str
) -> DataFrame:
    return (
        df
        .withColumn(
            "_arquivo_origem",
            F.lit(nome_arquivo)
        )
        .withColumn(
            "_timestamp_ingestao",
            F.current_timestamp()
        )
    )

# Carregando arquivos csv tudo string
def ler_csv_raw(
    nome_arquivo: str,
    separador: str
) -> DataFrame:

    df = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("sep", separador)
        .option("encoding", "UTF-8")
        .option("inferSchema", "false")
        .option("multiLine", "true")
        .option("quote", '"')
        .option("escape", '"')
        .option("mode", "PERMISSIVE")
        .load(caminho_landing(nome_arquivo))
    )

    return adicionar_metadados(
        df,
        nome_arquivo
    )

# Carregando arquivos JSON & NDJSON preservando a estrutura
def ler_json_raw(
    nome_arquivo: str,
    multiline: bool
) -> DataFrame:

    df = (
        spark.read
        .format("json")
        .option(
            "multiLine",
            str(multiline).lower()
        )
        .option("primitivesAsString", "true")
        .option("dropFieldIfAllNull", "false")
        .option("mode", "PERMISSIVE")
        .load(caminho_landing(nome_arquivo))
    )

    return adicionar_metadados(
        df,
        nome_arquivo
    )

# Carregando excel e convertendo para Spark.
def ler_excel_raw(
    nome_arquivo: str,
    nome_planilha=0
) -> DataFrame:
    caminho = caminho_landing(nome_arquivo)

    try:
        arquivo_excel = pd.ExcelFile(caminho)

    except ImportError as erro:
        raise RuntimeError(
            "A dependência não está disponível, faça o pip do openpyxl"
        ) from erro

    print(
        f"{nome_arquivo} | "
        f"planilhas: {arquivo_excel.sheet_names}"
    )

    pdf = pd.read_excel(
        caminho,
        sheet_name=nome_planilha,
        dtype=object
    )

    # Limpeando nomes das colunas
    pdf.columns = [
        str(coluna).strip()
        for coluna in pdf.columns
    ]

    registros = [
        tuple(
            None if pd.isna(valor) else str(valor)
            for valor in linha
        )
        for linha in pdf.itertuples(
            index=False,
            name=None
        )
    ]

    schema = StructType([
        StructField(
            coluna,
            StringType(),
            True
        )
        for coluna in pdf.columns
    ])

    df = spark.createDataFrame(
        registros,
        schema=schema
    )

    return adicionar_metadados(
        df,
        nome_arquivo
    )

In [0]:
# ============================================================
# CARREGAMENTO DAS FONTES
# ============================================================

fontes_raw = {
    "pedidos_cabecalho": ler_csv_raw(
        "erp_pedidos_cabecalho_2025.csv",
        separador=";"
    ),

    "pedidos_itens": ler_csv_raw(
        "erp_pedidos_itens_2025.csv",
        separador=","
    ),

    "vendedores": ler_csv_raw(
        "vendedores.csv",
        separador=";"
    ),

    "regioes": ler_csv_raw(
        "legado_regioes_pipe.txt",
        separador="|"
    ),

    "produtos": ler_json_raw(
        "cadastro_produtos_api_dump.json",
        multiline=True
    ),

    "entregas": ler_json_raw(
        "logistica_entregas.json",
        multiline=True
    ),

    "ocorrencias": ler_json_raw(
        "atendimento_ocorrencias.ndjson",
        multiline=False
    ),

    "clientes": ler_excel_raw(
        "crm_clientes_export.xlsx",
        nome_planilha=0
    ),

    "canais": ler_excel_raw(
        "comercial_canais.xlsx",
        nome_planilha="canais"
    )
}

print("\nFontes carregadas com sucesso:")

for nome_fonte in fontes_raw:
    print(f"- {nome_fonte}")

print(
    f"\nTotal de fontes carregadas: "
    f"{len(fontes_raw)}"
)

In [0]:
# ============================================================
# INVENTÁRIO DAS FONTES
# ============================================================

inventario = []

# Serializando cada registro para comparar e também em fontes com structs e arrays
for nome_fonte, df in fontes_raw.items():

    quantidade_registros = df.count()
    quantidade_colunas = len(df.columns)

    df_hash = df.select(
        F.sha2(
            F.to_json(
                F.struct(
                    *[
                        F.col(coluna)
                        for coluna in df.columns
                    ]
                )
            ),
            256
        ).alias("_hash_registro")
    )

    registros_distintos = (
        df_hash
        .select("_hash_registro")
        .distinct()
        .count()
    )

    duplicados_exatos = (
        quantidade_registros
        - registros_distintos
    )

    inventario.append(
        (
            nome_fonte,
            quantidade_registros,
            quantidade_colunas,
            registros_distintos,
            duplicados_exatos
        )
    )

# dict para inventario
schema_inventario = """
    fonte STRING,
    quantidade_registros LONG,
    quantidade_colunas LONG,
    registros_distintos LONG,
    duplicados_exatos LONG
"""

# inventario
df_inventario = (
    spark.createDataFrame(
        inventario,
        schema=schema_inventario
    )
    .orderBy("fonte")
)

display(df_inventario)

In [0]:
# ============================================================
# VALIDAR FONTES QUE ESTIVEREM VAZIAS
# ============================================================

fontes_vazias = (
    df_inventario
    .filter(
        F.col("quantidade_registros") == 0
    )
    .select("fonte")
    .collect()
)

if fontes_vazias:
    nomes = [
        linha["fonte"]
        for linha in fontes_vazias
    ]

    raise RuntimeError(
        "As seguintes fontes foram carregadas sem registros: \n- "
        + "\n- ".join(nomes)
    )

print(
    "Validação concluída, todas as fontes possuem registros!"
)

# Perfil inicial de qualidade dos dados

Nesta etapa será avaliado:

- schemas identificados nas fontes;
- tipos de dados recebidos;
- campos nulos e vazios;
- duplicidades nas chaves de negócio;
- inconsistências que deverão ser tratadas posteriormente na camada Silver.

As métricas desta seção representam os dados brutos, antes de qualquer padronização ou correção.

In [0]:
# ============================================================
# CATÁLOGO DE COLUNAS DAS FONTES
# ============================================================

catalogo_colunas = []

for nome_fonte, df in fontes_raw.items():

    for posicao, campo in enumerate(
        df.schema.fields,
        start=1
    ):
        catalogo_colunas.append(
            (
                nome_fonte,
                posicao,
                campo.name,
                campo.dataType.simpleString(),
                campo.nullable,
                campo.name.startswith("_")
            )
        )

schema_catalogo = """
    fonte STRING,
    posicao INT,
    coluna STRING,
    tipo_recebido STRING,
    permite_nulo BOOLEAN,
    coluna_metadado BOOLEAN
"""

df_catalogo_colunas = (
    spark.createDataFrame(
        catalogo_colunas,
        schema=schema_catalogo
    )
    .orderBy(
        "fonte",
        "posicao"
    )
)

display(df_catalogo_colunas)

In [0]:
# ============================================================
# SCHEMAS IDENTIFICADOS
# ============================================================

for nome_fonte, df in fontes_raw.items():

    print("\n" + "=" * 100)
    print(f"FONTE: {nome_fonte}")
    print("=" * 100)

    df.printSchema()

In [0]:
# ============================================================
# AMOSTRA DOS DADOS RECEBIDOS
# ============================================================

for nome_fonte, df in fontes_raw.items():

    print("\n" + "=" * 100)
    print(f"AMOSTRA DA FONTE: {nome_fonte}")
    print("=" * 100)

    display(
        df.limit(5)
    )

In [0]:
# ============================================================
# PERFIL DE NULOS E CAMPOS VAZIOS
# ============================================================

from pyspark.sql.types import StringType

perfil_ausencia = []

for nome_fonte, df in fontes_raw.items():

    quantidade_registros = df.count()

    campos_analisados = [
        campo
        for campo in df.schema.fields
        if not campo.name.startswith("_")
    ]

    agregacoes = []
    mapa_aliases = []

    for indice, campo in enumerate(campos_analisados):

        coluna = F.col(f"`{campo.name}`")

        alias_nulos = f"nulos_{indice}"

        agregacoes.append(
            F.sum(
                F.when(
                    coluna.isNull(),
                    1
                ).otherwise(0)
            ).cast("long").alias(alias_nulos)
        )

        alias_vazios = None

        if isinstance(campo.dataType, StringType):

            alias_vazios = f"vazios_{indice}"

            agregacoes.append(
                F.sum(
                    F.when(
                        coluna.isNotNull()
                        & (F.trim(coluna) == ""),
                        1
                    ).otherwise(0)
                ).cast("long").alias(alias_vazios)
            )

        mapa_aliases.append(
            (
                campo.name,
                campo.dataType.simpleString(),
                alias_nulos,
                alias_vazios
            )
        )

    resultado = (
        df
        .agg(*agregacoes)
        .first()
        .asDict()
    )

    for (
        nome_coluna,
        tipo_coluna,
        alias_nulos,
        alias_vazios
    ) in mapa_aliases:

        quantidade_nulos = (
            resultado.get(alias_nulos) or 0
        )

        quantidade_vazios = (
            resultado.get(alias_vazios) or 0
            if alias_vazios
            else 0
        )

        total_ausente = (
            quantidade_nulos
            + quantidade_vazios
        )

        percentual_ausente = (
            round(
                total_ausente
                / quantidade_registros
                * 100,
                2
            )
            if quantidade_registros > 0
            else 0.0
        )

        perfil_ausencia.append(
            (
                nome_fonte,
                nome_coluna,
                tipo_coluna,
                quantidade_registros,
                quantidade_nulos,
                quantidade_vazios,
                total_ausente,
                percentual_ausente
            )
        )

schema_ausencia = """
    fonte STRING,
    coluna STRING,
    tipo_coluna STRING,
    quantidade_registros LONG,
    quantidade_nulos LONG,
    quantidade_vazios LONG,
    total_ausente LONG,
    percentual_ausente DOUBLE
"""

df_perfil_ausencia = (
    spark.createDataFrame(
        perfil_ausencia,
        schema=schema_ausencia
    )
    .orderBy(
        F.desc("percentual_ausente"),
        "fonte",
        "coluna"
    )
)

display(df_perfil_ausencia)

In [0]:
# ============================================================
# CAMPOS COM AUSÊNCIA
# ============================================================

display(
    df_perfil_ausencia
    .filter(
        F.col("total_ausente") > 0
    )
    .orderBy(
        F.desc("percentual_ausente"),
        "fonte",
        "coluna"
    )
)

In [0]:
# ============================================================
# CHAVES DE NEGÓCIO IDENTIFICADAS
# ============================================================

CHAVES_NEGOCIO = {
    "pedidos_cabecalho": [
        "order_id"
    ],

    "pedidos_itens": [
        "order_id",
        "item_seq"
    ],

    "vendedores": [
        "seller_id"
    ],

    "regioes": [
        "regional_code"
    ],

    "produtos": [
        "product.product_id"
    ],

    "entregas": [
        "delivery_id"
    ],

    "ocorrencias": [
        "ticket_id"
    ],

    "clientes": [
        "customer_id"
    ],

    "canais": [
        "id_canal"
    ]
}

for fonte, chaves in CHAVES_NEGOCIO.items():
    print(
        f"{fonte}: "
        f"{', '.join(chaves)}"
    )

In [0]:
# ============================================================
# DUPLICIDADES NAS CHAVES
# ============================================================

from functools import reduce

resultado_duplicidades = []

for nome_fonte, caminhos_chaves in CHAVES_NEGOCIO.items():

    df = fontes_raw[nome_fonte]

    aliases_chaves = [
        f"chave_{indice}"
        for indice in range(len(caminhos_chaves))
    ]

    df_chaves = df.select(
        *[
            F.col(caminho)
            .cast("string")
            .alias(alias)
            for caminho, alias
            in zip(
                caminhos_chaves,
                aliases_chaves
            )
        ]
    )

    condicoes_validas = [
        F.col(alias).isNotNull()
        & (F.trim(F.col(alias)) != "")
        for alias in aliases_chaves
    ]

    condicao_chave_valida = reduce(
        lambda esquerda, direita:
            esquerda & direita,
        condicoes_validas
    )

    quantidade_total = df_chaves.count()

    quantidade_chave_ausente = (
        df_chaves
        .filter(~condicao_chave_valida)
        .count()
    )

    df_chaves_validas = (
        df_chaves
        .filter(condicao_chave_valida)
    )

    # Duplicidade
    duplicidades_raw = (
        df_chaves_validas
        .groupBy(*aliases_chaves)
        .count()
        .filter(F.col("count") > 1)
    )

    grupos_duplicados_raw = (
        duplicidades_raw.count()
    )

    registros_excedentes_raw = (
        duplicidades_raw
        .select(
            F.sum(
                F.col("count") - 1
            ).alias("excedentes")
        )
        .first()["excedentes"]
        or 0
    )

    # Duplicidade após normalização
    df_chaves_normalizadas = (
        df_chaves_validas
        .select(
            *[
                F.upper(
                    F.trim(
                        F.col(alias)
                    )
                ).alias(alias)
                for alias in aliases_chaves
            ]
        )
    )

    duplicidades_normalizadas = (
        df_chaves_normalizadas
        .groupBy(*aliases_chaves)
        .count()
        .filter(F.col("count") > 1)
    )

    grupos_duplicados_normalizados = (
        duplicidades_normalizadas.count()
    )

    registros_excedentes_normalizados = (
        duplicidades_normalizadas
        .select(
            F.sum(
                F.col("count") - 1
            ).alias("excedentes")
        )
        .first()["excedentes"]
        or 0
    )

    resultado_duplicidades.append(
        (
            nome_fonte,
            " + ".join(caminhos_chaves),
            quantidade_total,
            quantidade_chave_ausente,
            grupos_duplicados_raw,
            registros_excedentes_raw,
            grupos_duplicados_normalizados,
            registros_excedentes_normalizados
        )
    )

schema_duplicidades = """
    fonte STRING,
    chave_negocio STRING,
    quantidade_registros LONG,
    registros_chave_ausente LONG,
    grupos_duplicados_raw LONG,
    registros_excedentes_raw LONG,
    grupos_duplicados_normalizados LONG,
    registros_excedentes_normalizados LONG
"""

df_duplicidades_chave = (
    spark.createDataFrame(
        resultado_duplicidades,
        schema=schema_duplicidades
    )
    .orderBy("fonte")
)

display(df_duplicidades_chave)

In [0]:
# ============================================================
# VARREDURA DOS REGISTROS DUPLICADOS POR CHAVE
# ============================================================

from functools import reduce

# Verificando todos os registros cujas chaves de negócio aparecem  mais de uma vez
def obter_registros_duplicados(
    df: DataFrame,
    caminhos_chaves: list[str]
) -> DataFrame:
    aliases = [
        f"_chave_normalizada_{indice}"
        for indice in range(len(caminhos_chaves))
    ]

    df_com_chaves = df

    for caminho, alias in zip(caminhos_chaves, aliases):
        df_com_chaves = df_com_chaves.withColumn(
            alias,
            F.upper(
                F.trim(
                    F.col(caminho).cast("string")
                )
            )
        )

    condicoes_validas = [
        F.col(alias).isNotNull()
        & (F.col(alias) != "")
        for alias in aliases
    ]

    condicao_valida = reduce(
        lambda esquerda, direita: esquerda & direita,
        condicoes_validas
    )

    grupos_duplicados = (
        df_com_chaves
        .filter(condicao_valida)
        .groupBy(*aliases)
        .count()
        .filter(F.col("count") > 1)
    )

    return (
        df_com_chaves.alias("dados")
        .join(
            grupos_duplicados.alias("duplicados"),
            on=aliases,
            how="inner"
        )
        .orderBy(*aliases)
    )

fontes_com_duplicidade = [
    linha["fonte"]
    for linha in (
        df_duplicidades_chave
        .filter(
            F.col("registros_excedentes_normalizados") > 0
        )
        .select("fonte")
        .collect()
    )
]

duplicados_detalhados = {}

for nome_fonte in fontes_com_duplicidade:

    print("\n" + "=" * 100)
    print(f"DUPLICIDADES DA FONTE: {nome_fonte}")
    print("=" * 100)

    df_duplicados = obter_registros_duplicados(
        df=fontes_raw[nome_fonte],
        caminhos_chaves=CHAVES_NEGOCIO[nome_fonte]
    )

    duplicados_detalhados[nome_fonte] = df_duplicados

    display(df_duplicados)

## Conclusões sobre duplicidades

A análise demonstrou que as duplicidades não possuem uma única natureza

Oque identifiquei:

- registros completamente idênticos;
- versões diferentes da mesma chave com datas de atualização;
- registros com campos inválidos ou incompletos;
- conflitos de relacionamento com chaves inexistentes;
- diferenças apenas de representação ou padronização;

Na camada Silver, a resolução será realizada por regras específicas para cada entidade, utilizando, quando disponível:

1. maior data de atualização;
2. validade dos campos;
3. registros mais completos;
4. integridade referencial;
5. padronização;
6. regra documentada quando não houver informação suficiente.

Os registros que vou descartar da versão consolidada vão permanecer rastreáveis por meio de indicadores de qualidade ou área de quarentena

# Validação de formatos e relacionamentos

Nesta etapa check de inconsistências relacionadas:

- datas em formatos diferentes ou inválidos;
- campos monetarios e quantitativos;
- variações de texto;
- referências a entidades;
- possíveis impactos na modelagem analítica.

O foco é mensurar e documentar os problemas encontrados

In [0]:
# ============================================================
# VALIDAÇÃO DOS CAMPOS DE DATA
# ============================================================

FORMATOS_DATA = [
    "yyyy-MM-dd HH:mm:ss",
    "yyyy-MM-dd'T'HH:mm:ss",
    "yyyy/MM/dd HH:mm:ss",
    "yyyy/MM/dd HH:mm",
    "dd/MM/yyyy HH:mm:ss",
    "dd/MM/yyyy HH:mm",
    "yyyy-MM-dd",
    "yyyy/MM/dd",
    "dd/MM/yyyy"
]

# Converter uma coluna utilizando os formatos de data identificados nas fontes
def converter_timestamp_flexivel(caminho_coluna: str):
    coluna_string = F.trim(
        F.col(caminho_coluna).cast("string")
    )

    conversoes = [
        F.try_to_timestamp(
            coluna_string,
            F.lit(formato)
        )
        for formato in FORMATOS_DATA
    ]

    return F.coalesce(*conversoes)


CAMPOS_DATA = {
    "pedidos_cabecalho": [
        "order_date",
        "promised_date",
        "last_update"
    ],
    "vendedores": [
        "hire_date"
    ],
    "clientes": [
        "data_cadastro",
        "updated_at"
    ],
    "produtos": [
        "updated_at"
    ],
    "entregas": [
        "timestamps.shipped_at",
        "timestamps.delivered_at"
    ],
    "ocorrencias": [
        "created_at"
    ]
}

resultado_datas = []

for nome_fonte, campos in CAMPOS_DATA.items():

    df = fontes_raw[nome_fonte]
    quantidade_registros = df.count()

    for caminho_coluna in campos:

        valor_original = F.trim(
            F.col(caminho_coluna).cast("string")
        )

        df_validacao = df.select(
            valor_original.alias("valor_original"),
            converter_timestamp_flexivel(
                caminho_coluna
            ).alias("valor_convertido")
        )

        quantidade_ausentes = (
            df_validacao
            .filter(
                F.col("valor_original").isNull()
                | (F.col("valor_original") == "")
            )
            .count()
        )

        quantidade_invalidos = (
            df_validacao
            .filter(
                F.col("valor_original").isNotNull()
                & (F.col("valor_original") != "")
                & F.col("valor_convertido").isNull()
            )
            .count()
        )

        resultado_datas.append(
            (
                nome_fonte,
                caminho_coluna,
                quantidade_registros,
                quantidade_ausentes,
                quantidade_invalidos,
                round(
                    quantidade_invalidos
                    / quantidade_registros
                    * 100,
                    2
                )
                if quantidade_registros > 0
                else 0.0
            )
        )

schema_datas = """
    fonte STRING,
    campo STRING,
    quantidade_registros LONG,
    valores_ausentes LONG,
    valores_invalidos LONG,
    percentual_invalidos DOUBLE
"""

df_validacao_datas = (
    spark.createDataFrame(
        resultado_datas,
        schema=schema_datas
    )
    .orderBy(
        F.desc("valores_invalidos"),
        "fonte",
        "campo"
    )
)

display(df_validacao_datas)

In [0]:
# ============================================================
# VALORES INVÁLIDOS
# ============================================================

for nome_fonte, campos in CAMPOS_DATA.items():

    df = fontes_raw[nome_fonte]

    for caminho_coluna in campos:

        invalidos = (
            df
            .select(
                F.col(caminho_coluna)
                .cast("string")
                .alias("valor_original")
            )
            .withColumn(
                "valor_convertido",
                converter_timestamp_flexivel(
                    "valor_original"
                )
            )
            .filter(
                F.col("valor_original").isNotNull()
                & (F.trim(F.col("valor_original")) != "")
                & F.col("valor_convertido").isNull()
            )
            .groupBy("valor_original")
            .count()
            .orderBy(
                F.desc("count"),
                "valor_original"
            )
        )

        if invalidos.limit(1).count() > 0:
            print(
                f"\nValores de data inválidos: "
                f"{nome_fonte}.{caminho_coluna}"
            )

            display(invalidos)

## Resultado da validação de datas

- pedidos_cabecalho.order_date = 2025-13-40
- entregas.timestamps.shipped_at = 31/02/2025 10:00

Esses valores não serão corrigidos, pois não há informação suficiente para determinar a data correta

In [0]:
# ============================================================
# VALIDAÇÃO DE CAMPOS NUMÉRICOS E MONETÁRIOS
# ============================================================

# Função para Converter valores com ponto ou vírgula decimal
def converter_decimal_flexivel(caminho_coluna: str):
    valor_original = F.trim(
        F.col(caminho_coluna).cast("string")
    )

    # Remove símbolos, textos e espaços.
    valor_limpo = F.regexp_replace(
        valor_original,
        r"[^0-9,.\-]",
        ""
    )

    # Quando existe vírgula, considera-a separador decimal.
    valor_padronizado = (
        F.when(
            valor_limpo.isNull()
            | (valor_limpo == ""),
            F.lit(None).cast("string")
        )
        .when(
            F.instr(valor_limpo, ",") > 0,
            F.regexp_replace(
                F.regexp_replace(
                    valor_limpo,
                    r"\.",
                    ""
                ),
                ",",
                "."
            )
        )
        .otherwise(valor_limpo)
    )

    return valor_padronizado.try_cast(
        "decimal(18,4)"
    )


CAMPOS_NUMERICOS = {
    "pedidos_cabecalho": [
        "gross_amount",
        "discount_amount",
        "net_amount"
    ],
    "pedidos_itens": [
        "quantity",
        "unit_price",
        "total_item"
    ],
    "produtos": [
        "pricing.list_price"
    ],
    "entregas": [
        "cost"
    ]
}

resultado_numericos = []

for nome_fonte, campos in CAMPOS_NUMERICOS.items():

    df = fontes_raw[nome_fonte]
    quantidade_registros = df.count()

    for caminho_coluna in campos:

        valor_original = F.trim(
            F.col(caminho_coluna).cast("string")
        )

        df_validacao = df.select(
            valor_original.alias("valor_original"),
            converter_decimal_flexivel(
                caminho_coluna
            ).alias("valor_convertido")
        )

        quantidade_ausentes = (
            df_validacao
            .filter(
                F.col("valor_original").isNull()
                | (F.col("valor_original") == "")
            )
            .count()
        )

        quantidade_invalidos = (
            df_validacao
            .filter(
                F.col("valor_original").isNotNull()
                & (F.col("valor_original") != "")
                & F.col("valor_convertido").isNull()
            )
            .count()
        )

        quantidade_negativos = (
            df_validacao
            .filter(
                F.col("valor_convertido") < 0
            )
            .count()
        )

        resultado_numericos.append(
            (
                nome_fonte,
                caminho_coluna,
                quantidade_registros,
                quantidade_ausentes,
                quantidade_invalidos,
                quantidade_negativos
            )
        )

schema_numericos = """
    fonte STRING,
    campo STRING,
    quantidade_registros LONG,
    valores_ausentes LONG,
    valores_invalidos LONG,
    valores_negativos LONG
"""

df_validacao_numericos = (
    spark.createDataFrame(
        resultado_numericos,
        schema=schema_numericos
    )
    .orderBy(
        F.desc("valores_invalidos"),
        "fonte",
        "campo"
    )
)


display(df_validacao_numericos)

In [0]:
# ============================================================
# VALORES INVÁLIDOS
# ============================================================

for nome_fonte, campos in CAMPOS_NUMERICOS.items():

    df = fontes_raw[nome_fonte]

    for caminho_coluna in campos:

        invalidos = (
            df
            .select(
                F.col(caminho_coluna)
                .cast("string")
                .alias("valor_original")
            )
            .withColumn(
                "valor_convertido",
                converter_decimal_flexivel(
                    "valor_original"
                )
            )
            .filter(
                F.col("valor_original").isNotNull()
                & (F.trim(F.col("valor_original")) != "")
                & F.col("valor_convertido").isNull()
            )
            .groupBy("valor_original")
            .count()
            .orderBy(
                F.desc("count"),
                "valor_original"
            )
        )

        if invalidos.limit(1).count() > 0:
            print(
                f"\nValores numéricos inválidos: "
                f"{nome_fonte}.{caminho_coluna}"
            )

            display(invalidos)

## Resultado da validação de campos numéricos e monetários

- pedidos_cabecalho.gross_amount = N/A — 1 registro
- produtos.pricing.list_price = N/A — 1 registro
- entregas.cost = unknown — 10 registros

Esses valores não vão ser convertidos para zero, pois isso alteraria indevidamente os indicadores

In [0]:
# ============================================================
# MAPEAMENTO DOS CAMPOS CATEGÓRICOS
# ============================================================

CAMPOS_CATEGORICOS = {
    "pedidos_cabecalho": [
        "status_order"
    ],
    "pedidos_itens": [
        "item_status"
    ],
    "vendedores": [
        "status"
    ],
    "clientes": [
        "segmento",
        "porte",
        "estado",
        "status_cliente"
    ],
    "canais": [
        "nome_canal",
        "tipo_canal",
        "ativo"
    ],
    "regioes": [
        "regional_code",
        "state",
        "active_flag"
    ],
    "produtos": [
        "product.category",
        "product.subcategory",
        "product.status",
        "attributes.family"
    ],
    "entregas": [
        "delivery_status",
        "carrier.mode",
        "destination.state"
    ],
    "ocorrencias": [
        "event_type",
        "severity",
        "status"
    ]
}

for nome_fonte, campos in CAMPOS_CATEGORICOS.items():
    df = fontes_raw[nome_fonte]

    for caminho_coluna in campos:

        print(
            f"\nDOMÍNIO: "
            f"{nome_fonte}.{caminho_coluna}"
        )

        display(
            df
            .select(
                F.col(caminho_coluna)
                .cast("string")
                .alias("valor_original")
            )
            .withColumn(
                "valor_normalizado",
                F.upper(
                    F.trim(
                        F.col("valor_original")
                    )
                )
            )
            .groupBy(
                "valor_original",
                "valor_normalizado"
            )
            .count()
            .orderBy(
                F.desc("count"),
                "valor_normalizado"
            )
        )

## Resultado da análise dos campos categóricos

- diferenças entre letras maiúsculas e minúsculas;
- presença ou ausência de acentuação;
- uso alternado de espaços e underscores;
- estados informados por sigla, nome completo ou abreviação;
- códigos de região fora do padrão esperado;
- valores nulos em atributos;
- diferenças de grafia para o mesmo status ou categoria.

Exemplos

- em separacao, EM SEPARAÇÃO e EM_SEPARACAO
- Ativo, ativo e ATIVO
- Rodoviário e rodoviario
- E-commerce e ecommerce
- SP, São Paulo e sao paulo
- Santa Catarina, Sta Catarina, S. Catarina e SC
- código regional S e sul

Essas variações serão consolidadas na camada Silver por meio de regras de padronização e tabelas.
Agora sobre os valores nulos não serão substituídos automaticamente por uma categoria existente, quando necessário para consumo analítico, vão ser representados como NAO_INFORMADO, mantendo também um indicador de qualidade.

In [0]:
# ============================================================
# VALIDAÇÃO DOS RELACIONAMENTOS
# ============================================================

RELACIONAMENTOS = [
    (
        "pedidos_cliente",
        "pedidos_cabecalho",
        "customer_code",
        "clientes",
        "customer_id"
    ),
    (
        "pedidos_vendedor",
        "pedidos_cabecalho",
        "seller_id",
        "vendedores",
        "seller_id"
    ),
    (
        "vendedor_canal",
        "vendedores",
        "canal_id",
        "canais",
        "id_canal"
    ),
    (
        "vendedor_regiao",
        "vendedores",
        "regional_code",
        "regioes",
        "regional_code"
    ),
    (
        "item_pedido",
        "pedidos_itens",
        "order_id",
        "pedidos_cabecalho",
        "order_id"
    ),
    (
        "item_produto",
        "pedidos_itens",
        "product_code",
        "produtos",
        "product.product_id"
    ),
    (
        "entrega_pedido",
        "entregas",
        "order_ref",
        "pedidos_cabecalho",
        "order_id"
    ),
    (
        "ocorrencia_pedido",
        "ocorrencias",
        "order_id",
        "pedidos_cabecalho",
        "order_id"
    )
]

def normalizar_chave(caminho_coluna: str):
    return F.upper(
        F.trim(
            F.col(caminho_coluna).cast("string")
        )
    )


resultado_integridade = []
chaves_unicas_detalhadas = {}

for (
    relacionamento,
    fonte_filha,
    chave_filha,
    fonte_pai,
    chave_pai
) in RELACIONAMENTOS:

    df_filha = (
        fontes_raw[fonte_filha]
        .select(
            normalizar_chave(
                chave_filha
            ).alias("_chave")
        )
    )

    df_pai = (
        fontes_raw[fonte_pai]
        .select(
            normalizar_chave(
                chave_pai
            ).alias("_chave")
        )
        .filter(
            F.col("_chave").isNotNull()
            & (F.col("_chave") != "")
        )
        .distinct()
    )

    chaves_ausentes = (
        df_filha
        .filter(
            F.col("_chave").isNull()
            | (F.col("_chave") == "")
        )
        .count()
    )

    df_filha_valida = (
        df_filha
        .filter(
            F.col("_chave").isNotNull()
            & (F.col("_chave") != "")
        )
    )

    df_unicas = (
        df_filha_valida
        .join(
            df_pai,
            on="_chave",
            how="left_anti"
        )
    )

    registros_unicos = df_unicas.count()

    chaves_unicas_distintas = (
        df_unicas
        .select("_chave")
        .distinct()
        .count()
    )

    chaves_unicas_detalhadas[
        relacionamento
    ] = (
        df_unicas
        .groupBy("_chave")
        .count()
        .orderBy(
            F.desc("count"),
            "_chave"
        )
    )

    resultado_integridade.append(
        (
            relacionamento,
            fonte_filha,
            chave_filha,
            fonte_pai,
            chave_pai,
            df_filha.count(),
            chaves_ausentes,
            registros_unicos,
            chaves_unicas_distintas
        )
    )

schema_integridade = """
    relacionamento STRING,
    fonte_filha STRING,
    chave_filha STRING,
    fonte_pai STRING,
    chave_pai STRING,
    quantidade_registros LONG,
    chaves_ausentes LONG,
    registros_unicos LONG,
    chaves_unicas_distintas LONG
"""

df_integridade_referencial = (
    spark.createDataFrame(
        resultado_integridade,
        schema=schema_integridade
    )
    .orderBy(
        F.desc("registros_unicos"),
        "relacionamento"
    )
)

display(df_integridade_referencial)

## Resultado da validação de relacionamento

Esses registros não serão descartados automaticamente, pois ainda representam eventos válidos das fontes transacionais, essa estratégia evita perda de fatos transacionais e permite que o analista identifique a parcela dos dados afetada por problemas cadastrais   

In [0]:
# ============================================================
# CONSISTÊNCIA FINANCEIRA DOS PEDIDOS
# ============================================================

df_validacao_financeira_pedidos = (
    fontes_raw["pedidos_cabecalho"]
    .select(
        F.col("order_id"),
        F.col("gross_amount").alias("gross_amount_original"),
        F.col("discount_amount").alias("discount_amount_original"),
        F.col("net_amount").alias("net_amount_original"),
        converter_decimal_flexivel(
            "gross_amount"
        ).alias("gross_amount"),
        converter_decimal_flexivel(
            "discount_amount"
        ).alias("discount_amount"),
        converter_decimal_flexivel(
            "net_amount"
        ).alias("net_amount")
    )
    .withColumn(
        "net_amount_calculado",
        F.col("gross_amount")
        - F.coalesce(
            F.col("discount_amount"),
            F.lit(0).cast("decimal(18,4)")
        )
    )
    .withColumn(
        "diferenca_financeira",
        F.abs(
            F.col("net_amount")
            - F.col("net_amount_calculado")
        )
    )
    .withColumn(
        "dq_divergencia_financeira",
        F.when(
            F.col("gross_amount").isNull()
            | F.col("net_amount").isNull(),
            F.lit(None).cast("boolean")
        )
        .otherwise(
            F.col("diferenca_financeira") > F.lit(0.01)
        )
    )
)

resumo_financeiro_pedidos = (
    df_validacao_financeira_pedidos
    .agg(
        F.count("*").alias("quantidade_registros"),
        F.sum(
            F.when(
                F.col("dq_divergencia_financeira") == True,
                1
            ).otherwise(0)
        ).alias("registros_divergentes"),
        F.sum(
            F.when(
                F.col("dq_divergencia_financeira").isNull(),
                1
            ).otherwise(0)
        ).alias("registros_nao_avaliados")
    )
)

display(resumo_financeiro_pedidos)

In [0]:
# ============================================================
# DIVERGÊNCIAS FINANCEIRA DOS PEDIDOS
# ============================================================

display(
    df_validacao_financeira_pedidos
    .filter(
        F.col("dq_divergencia_financeira") == True
    )
    .orderBy(
        F.desc("diferenca_financeira")
    )
)

## Resultado da consistência financeira dos pedidos

- 12 pedidos com divergência superior à tolerância de R$ 0,01
- 1 pedido não avaliado por possuir valor monetário não conversível
- 390 pedidos consistentes com a regra

A definição do valor final para consumo analítico será realizada conforme a comparação com os totais dos itens dos pedidos

In [0]:
# ============================================================
# CONSISTÊNCIA DOS ITENS DE PEDIDO
# ============================================================

df_validacao_financeira_itens = (
    fontes_raw["pedidos_itens"]
    .select(
        F.col("order_id"),
        F.col("item_seq"),
        F.col("product_code"),
        F.col("quantity").alias("quantity_original"),
        F.col("unit_price").alias("unit_price_original"),
        F.col("total_item").alias("total_item_original"),
        converter_decimal_flexivel(
            "quantity"
        ).alias("quantity"),
        converter_decimal_flexivel(
            "unit_price"
        ).alias("unit_price"),
        converter_decimal_flexivel(
            "total_item"
        ).alias("total_item")
    )
    .withColumn(
        "total_item_calculado",
        F.col("quantity")
        * F.col("unit_price")
    )
    .withColumn(
        "diferenca_item",
        F.abs(
            F.col("total_item")
            - F.col("total_item_calculado")
        )
    )
    .withColumn(
        "dq_divergencia_total_item",
        F.when(
            F.col("quantity").isNull()
            | F.col("unit_price").isNull()
            | F.col("total_item").isNull(),
            F.lit(None).cast("boolean")
        )
        .otherwise(
            F.col("diferenca_item") > F.lit(0.01)
        )
    )
)

resumo_financeiro_itens = (
    df_validacao_financeira_itens
    .agg(
        F.count("*").alias("quantidade_registros"),
        F.sum(
            F.when(
                F.col("dq_divergencia_total_item") == True,
                1
            ).otherwise(0)
        ).alias("registros_divergentes"),
        F.sum(
            F.when(
                F.col("dq_divergencia_total_item").isNull(),
                1
            ).otherwise(0)
        ).alias("registros_nao_avaliados")
    )
)

display(resumo_financeiro_itens)


display(
    df_validacao_financeira_itens
    .filter(
        F.col("dq_divergencia_total_item") == True
    )
    .orderBy(
        F.desc("diferenca_item")
    )
)

## Resultado da consistência financeira dos itens de pedido

Foram identificados 47 itens com divergência superior à tolerância de R$ 0,01, representando aproximadamente 4,72% dos registros da fonte, as diferenças identificadas variam entre R$ 0,50 e R$ 19,62, indicando que os casos não podem ser explicados apenas por arredondamento monetário.

In [0]:
# ============================================================
# CONSISTÊNCIA TEMPORAL
# ============================================================

df_validacao_datas_pedidos = (
    fontes_raw["pedidos_cabecalho"]
    .select(
        F.col("order_id"),
        F.col("order_date").alias("order_date_original"),
        F.col("promised_date").alias("promised_date_original"),
        converter_timestamp_flexivel(
            "order_date"
        ).alias("order_date"),
        converter_timestamp_flexivel(
            "promised_date"
        ).alias("promised_date")
    )
    .withColumn(
        "dq_promessa_anterior_pedido",
        F.when(
            F.col("order_date").isNull()
            | F.col("promised_date").isNull(),
            F.lit(None).cast("boolean")
        )
        .otherwise(
            F.col("promised_date")
            < F.col("order_date")
        )
    )
)

display(
    df_validacao_datas_pedidos
    .filter(
        F.col("dq_promessa_anterior_pedido") == True
    )
)

In [0]:
df_validacao_datas_entregas = (
    fontes_raw["entregas"]
    .select(
        F.col("delivery_id"),
        F.col("order_ref"),
        F.col("delivery_status"),
        F.col("timestamps.shipped_at")
        .alias("shipped_at_original"),
        F.col("timestamps.delivered_at")
        .alias("delivered_at_original"),
        converter_timestamp_flexivel(
            "timestamps.shipped_at"
        ).alias("shipped_at"),
        converter_timestamp_flexivel(
            "timestamps.delivered_at"
        ).alias("delivered_at")
    )
    .withColumn(
        "status_normalizado",
        F.upper(
            F.trim(
                F.col("delivery_status")
            )
        )
    )
    .withColumn(
        "dq_entrega_anterior_envio",
        F.when(
            F.col("shipped_at").isNull()
            | F.col("delivered_at").isNull(),
            F.lit(None).cast("boolean")
        )
        .otherwise(
            F.col("delivered_at")
            < F.col("shipped_at")
        )
    )
    .withColumn(
        "dq_entregue_sem_data",
        (
            F.col("status_normalizado") == "DELIVERED"
        )
        & F.col("delivered_at").isNull()
    )
    .withColumn(
        "dq_transito_com_data_entrega",
        (
            F.col("status_normalizado") == "IN_TRANSIT"
        )
        & F.col("delivered_at").isNotNull()
    )
)

display(
    df_validacao_datas_entregas
    .filter(
        (F.col("dq_entrega_anterior_envio") == True)
        | (F.col("dq_entregue_sem_data") == True)
        | (F.col("dq_transito_com_data_entrega") == True)
    )
)

## Resultado da consistência temporal

A validação temporal dos pedidos não identificou casos em que a data prometida fosse anterior à data de criação do pedido e também não foram encontrados registros válidos em que a data de entrega fosse anterior a data de envio.

Inconsistências entre os status logísticos e as datas informadas:

- 11 entregas possuem status DELIVERED, mas não apresentam data de entrega;
- 53 entregas possuem status IN_TRANSIT, embora já apresentem data de entrega;
- total de 64 registros com divergência entre status e eventos temporais.

Essas inconsistências podem indicar atraso na atualização do status operacional ou preenchimento incorreto das datas    

In [0]:
# ============================================================
# RESUMO CONSOLIDADO DA QUALIDADE DOS DADOS
# ============================================================

resumo_qualidade = []


def registrar_problema(
    categoria: str,
    fonte: str,
    campo_relacionamento: str,
    quantidade: int,
    severidade: str,
    descricao: str,
    tratamento_proposto: str
):
    # Registrar no resumo apenas problemas com quantidade maior que zero
    quantidade = int(quantidade or 0)

    if quantidade <= 0:
        return

    resumo_qualidade.append(
        (
            categoria,
            fonte,
            campo_relacionamento,
            quantidade,
            severidade,
            descricao,
            tratamento_proposto
        )
    )

In [0]:
#============================================================# 
# DUPLICIDADE POR CHAVE
# ============================================================

for linha in (
    df_duplicidades_chave
    .filter(
        F.col("registros_excedentes_normalizados") > 0
    )
    .collect()
):

    fonte = linha["fonte"]

    severidade = (
        "ALTA"
        if fonte in {
            "pedidos_cabecalho",
            "pedidos_itens",
            "entregas"
        }
        else "MEDIA"
    )

    registrar_problema(
        categoria="DUPLICIDADE_CHAVE",
        fonte=fonte,
        campo_relacionamento=linha["chave_negocio"],
        quantidade=linha[
            "registros_excedentes_normalizados"
        ],
        severidade=severidade,
        descricao=(
            "Registros excedentes após normalização, das chaves de negócio"
        ),
        tratamento_proposto=(
            "Aplicar regra de deduplicação específica por entidade, preservando rastreabilidade e registros com conflito"
        )
    )

In [0]:
#============================================================# 
# DATAS INVÁLIDAS
# ============================================================

for linha in (
    df_validacao_datas
    .filter(
        F.col("valores_invalidos") > 0
    )
    .collect()
):

    registrar_problema(
        categoria="DATA_INVALIDA",
        fonte=linha["fonte"],
        campo_relacionamento=linha["campo"],
        quantidade=linha["valores_invalidos"],
        severidade="ALTA",
        descricao=(
            "Valor preenchido que não representa uma data valida"
        ),
        tratamento_proposto=(
            "Preservar o valor original, converter o campo tratado para NULL e criar indicador de qualidade"
        )
    )

In [0]:
#=============================================================
# VALORES NÚMERICOS NÃO CONVERSIVEIS
# ============================================================

for linha in (
    df_validacao_numericos
    .filter(
        F.col("valores_invalidos") > 0
    )
    .collect()
):

    registrar_problema(
        categoria="NUMERICO_INVALIDO",
        fonte=linha["fonte"],
        campo_relacionamento=linha["campo"],
        quantidade=linha["valores_invalidos"],
        severidade="ALTA",
        descricao=(
            "Valor preenchido que não pode ser convertido para um tipo numérico"
        ),
        tratamento_proposto=(
            "Não converter para zero, preservar o valor original e gerar NULL no campo tratado e sinalizar a inconsistência"
        )
    )

In [0]:
#=============================================================
# INTEGRIDADE REFFERENCIAL
# ============================================================

for linha in df_integridade_referencial.collect():

    registrar_problema(
        categoria="CHAVE_AUSENTE",
        fonte=linha["fonte_filha"],
        campo_relacionamento=linha["relacionamento"],
        quantidade=linha["chaves_ausentes"],
        severidade="MEDIA",
        descricao=(
            f"Registros sem preenchimento da chave "
            f"{linha['chave_filha']}."
        ),
        tratamento_proposto=(
            "Preservar o registro, sinalizar ausência e utilizar registro técnico NAO_INFORMADO no modelo analítico"
        )
    )

    registrar_problema(
        categoria="CHAVE_UNICA",
        fonte=linha["fonte_filha"],
        campo_relacionamento=linha["relacionamento"],
        quantidade=linha["registros_unicos"],
        severidade="ALTA",
        descricao=(
            f"Chave preenchida sem correspondência na fonte "
            f"{linha['fonte_pai']}."
        ),
        tratamento_proposto=(
            "Preservar a fato transacional, marcar a chave como unica e associar a um registro técnico NAO_IDENTIFICADO"
        )
    )

In [0]:
#=============================================================
# CONSISTÊNCIA FINANCEIRA
# ============================================================

resumo_pedidos = (
    resumo_financeiro_pedidos
    .first()
    .asDict()
)

registrar_problema(
    categoria="DIVERGENCIA_FINANCEIRA",
    fonte="pedidos_cabecalho",
    campo_relacionamento=(
        "gross_amount - discount_amount = net_amount"
    ),
    quantidade=resumo_pedidos["registros_divergentes"],
    severidade="ALTA",
    descricao=(
        "Valor líquido informado divergente do valor bruto menos o desconto"
    ),
    tratamento_proposto=(
        "Preservar valores informados e calculados, registrar a diferença e definir o valor analítico após comparação com os itens do pedido"
    )
)

registrar_problema(
    categoria="REGISTRO_NAO_AVALIADO",
    fonte="pedidos_cabecalho",
    campo_relacionamento="consistencia_financeira",
    quantidade=resumo_pedidos["registros_nao_avaliados"],
    severidade="ALTA",
    descricao=(
        "Pedido não avaliado por ausência de valores"
    ),
    tratamento_proposto=(
        "Manter indicador de não avaliação e impedir que o registro, seja considerado consistente automaticamente"
    )
)


resumo_itens = (
    resumo_financeiro_itens
    .first()
    .asDict()
)

registrar_problema(
    categoria="DIVERGENCIA_FINANCEIRA",
    fonte="pedidos_itens",
    campo_relacionamento=(
        "quantity * unit_price = total_item"
    ),
    quantidade=resumo_itens["registros_divergentes"],
    severidade="ALTA",
    descricao=(
        "Total informado do item divergente do valor"
    ),
    tratamento_proposto=(
        "Registrar a diferença e comparar as somas com o cabeçalho do pedido"
    )
)

registrar_problema(
    categoria="REGISTRO_NAO_AVALIADO",
    fonte="pedidos_itens",
    campo_relacionamento="consistencia_financeira_item",
    quantidade=resumo_itens["registros_nao_avaliados"],
    severidade="MEDIA",
    descricao=(
        "Item não avaliado por ausência de quantidade, preço unitário ou total"
    ),
    tratamento_proposto=(
        "Manter o registro e sinalizar que sua consistência financeira não pôde ser validada"
    )
)

In [0]:
#=============================================================
# CONSISTÊNCIA TEMPORAL
# ============================================================

promessas_anteriores = (
    df_validacao_datas_pedidos
    .filter(
        F.col("dq_promessa_anterior_pedido") == True
    )
    .count()
)

entregas_anteriores_envio = (
    df_validacao_datas_entregas
    .filter(
        F.col("dq_entrega_anterior_envio") == True
    )
    .count()
)

entregues_sem_data = (
    df_validacao_datas_entregas
    .filter(
        F.col("dq_entregue_sem_data") == True
    )
    .count()
)

transito_com_data_entrega = (
    df_validacao_datas_entregas
    .filter(
        F.col("dq_transito_com_data_entrega") == True
    )
    .count()
)


registrar_problema(
    categoria="INCONSISTENCIA_TEMPORAL",
    fonte="pedidos_cabecalho",
    campo_relacionamento="promised_date < order_date",
    quantidade=promessas_anteriores,
    severidade="ALTA",
    descricao=(
        "Data prometida anterior à data de criação do pedido"
    ),
    tratamento_proposto=(
        "Preservar valores originais e sinalizar inconsistências"
    )
)

registrar_problema(
    categoria="INCONSISTENCIA_TEMPORAL",
    fonte="entregas",
    campo_relacionamento="delivered_at < shipped_at",
    quantidade=entregas_anteriores_envio,
    severidade="ALTA",
    descricao=(
        "Data de entrega anterior a data de envio"
    ),
    tratamento_proposto=(
        "Preservar valores originais e sinalizar inconsistências"
    )
)

registrar_problema(
    categoria="INCONSISTENCIA_STATUS_DATA",
    fonte="entregas",
    campo_relacionamento="DELIVERED sem delivered_at",
    quantidade=entregues_sem_data,
    severidade="ALTA",
    descricao=(
        "Entrega marcada como concluída sem data de entrega"
    ),
    tratamento_proposto=(
        "Manter status original e criar indicador de inconsistências"
    )
)

registrar_problema(
    categoria="INCONSISTENCIA_STATUS_DATA",
    fonte="entregas",
    campo_relacionamento="IN_TRANSIT com delivered_at",
    quantidade=transito_com_data_entrega,
    severidade="ALTA",
    descricao=(
        "Entrega em trânsito com data de entrega preenchida"
    ),
    tratamento_proposto=(
        "Manter status original e criar status analítico derivado se necessário para consumo"
    )
)

In [0]:
#=============================================================
# AUSÊNCIAS
# ============================================================

LIMITE_PERCENTUAL_AUSENCIA = 10.0

for linha in (
    df_perfil_ausencia
    .filter(
        F.col("percentual_ausente")
        >= LIMITE_PERCENTUAL_AUSENCIA
    )
    .collect()
):

    percentual = linha["percentual_ausente"]

    severidade = (
        "ALTA"
        if percentual >= 30
        else "MEDIA"
    )

    registrar_problema(
        categoria="AUSENCIA_RELEVANTE",
        fonte=linha["fonte"],
        campo_relacionamento=linha["coluna"],
        quantidade=linha["total_ausente"],
        severidade=severidade,
        descricao=(
            f"Campo com {percentual:.2f}% de valores nulos ou vazios"
        ),
        tratamento_proposto=(
            "Avaliar impacto por campo, não preencher métricas, usar valor técnico apenas em atributos dimensionais quando necessário"
        )
    )

In [0]:
#=============================================================
# CONSOLIDADO
# ============================================================

schema_resumo_qualidade = """
    categoria_problema STRING,
    fonte STRING,
    campo_relacionamento STRING,
    quantidade LONG,
    severidade STRING,
    descricao STRING,
    tratamento_proposto STRING
"""

df_resumo_qualidade = (
    spark.createDataFrame(
        resumo_qualidade,
        schema=schema_resumo_qualidade
    )
    .withColumn(
        "_ordem_severidade",
        F.when(
            F.col("severidade") == "ALTA",
            1
        )
        .when(
            F.col("severidade") == "MEDIA",
            2
        )
        .otherwise(3)
    )
    .orderBy(
        "_ordem_severidade",
        F.desc("quantidade"),
        "categoria_problema",
        "fonte"
    )
    .drop("_ordem_severidade")
)

df_resumo_qualidade.createOrReplaceTempView(
    "vw_resumo_qualidade_exploracao"
)

display(df_resumo_qualidade)

In [0]:
# ============================================================
# FINAL
# ============================================================

df_resumo_qualidade_executivo = (
    df_resumo_qualidade
    .groupBy(
        "severidade",
        "categoria_problema"
    )
    .agg(
        F.count("*").alias(
            "quantidade_regras_afetadas"
        ),
        F.sum("quantidade").alias(
            "soma_ocorrencias"
        ),
        F.countDistinct("fonte").alias(
            "fontes_afetadas"
        )
    )
    .withColumn(
        "_ordem_severidade",
        F.when(
            F.col("severidade") == "ALTA",
            1
        )
        .when(
            F.col("severidade") == "MEDIA",
            2
        )
        .otherwise(3)
    )
    .orderBy(
        "_ordem_severidade",
        F.desc("soma_ocorrencias")
    )
    .drop("_ordem_severidade")
)

display(df_resumo_qualidade_executivo)

# Conclusão da exploração das fontes

Foram exploradas nove fontes brutas, provenientes de arquivos CSV, TXT , Excel, JSON e NDJSON

A análise confirma que os dados possuem potencial para construção de um modelo analítico integrado, porém exigem tratamentos de qualidade antes de serem disponibilizados para consumo por BI

## Principais problemas encontrados

- duplicidades em chaves de negócio de diferentes entidades
- registros duplicados com conteúdos conflitantes
- datas em formatos variados e duas datas efetivamente inválidas
- valores monetários preenchidos com N/A e unknown
- inconsistências entre valores financeiros informados e recalculados
- variações de grafia, acentuação, caixa e representação em campos categóricos
- estados representados por sigla, nome completo e abreviações
- registros transacionais associados a clientes, produtos, pedidos ou canais inexistentes
- divergências entre status logístico e datas de entrega
- campos cadastrais com percentual relevante de ausência

## Impactos potenciais

Sem tratamento, esses problemas poderiam causar:

- duplicação de receita e volume
- cálculo incorreto de ticket médio
- segmentação incorreta por região, canal e produto
- subestimação ou superestimação de custos logísticos
- taxas de atraso e cancelamento inconsistentes
- perda de fatos transacionais durante relacionamentos
- divergências entre dashboards construídos por diferentes analistas

## Estratégia definida

A camada Bronze preservará os dados como recebidos, acrescentando metadados de rastreabilidade já a camada Silver será responsável por:

- padronizar chaves, datas, valores e domínios
- aplicar regras específicas de deduplicação
- preservar valores originais e criar versões tratadas
- criar indicadores de qualidade
- manter registros órfãos sem eliminar fatos transacionais
- direcionar conflitos não resolvidos para uma estrutura de inconsistências
- produzir entidades consolidadas e confiáveis

A camada Gold apresentará dimensões e fatos orientados ao consumo analítico, com granularidades e relacionamentos claramente documentados

## Limitações

Não existe informação suficiente para corrigir por inferência determinados valores inválidos ou conflitos entre versões, nesses casos a solução priorizará transparência, rastreabilidade e sinalização do problema, evitando correções silenciosas ou arbitrárias.